# 🔄 WineCycloPep — Example Notebook

**Step 3 of the Wine Peptidome series**  
De novo cyclic peptide discovery · FDR validation · GFN2-xTB 3D conformers · Metal chelation

---

This notebook demonstrates the full `wine_cyclopep.py` pipeline on **synthetic timsTOF PASEF data**
that realistically simulates wine lees fractions from *Saccharomyces cerevisiae* autolysis.

**Two sections:**
1. 🍷 **Wine lees context** — known DKPs and cyclic peptides from *S. cerevisiae* lees (Sémillon / Cabernet Sauvignon fractions)
2. 🧪 **Generic validation** — cyclo-RGD, cyclo-VAAG and mixed sequences for pipeline benchmarking

> **No real `.d` file required** — a `MockTimsTOF` class generates synthetic MS2 spectra with
> realistic signal/noise and ion coverage, allowing the full pipeline to run out-of-the-box.

---

### Requirements
```
pip install alphatims rdkit scipy tblite
```


## 0. Imports and setup

In [ ]:
import sys, os, warnings, time
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

# ── add parent directory to path if running from examples/ ───────────────────
sys.path.insert(0, str(Path(".").resolve().parent))
sys.path.insert(0, str(Path(".").resolve()))

# ── import wine_cyclopep functions directly ───────────────────────────────────
from wine_cyclopep import (
    # Mass / fragmentation
    cyclic_neutral_mass, linear_neutral_mass, canonical_cyclic,
    compositions_for_mass, all_permutations,
    bn_ions_all_rotations, residue_loss_ions,
    score_spectrum_vs_cyclic, confidence_tier,
    # 3D structure
    build_cyclic_smiles, generate_conformers,
    # Chelation
    predict_chelation,
    # Statistics
    generate_decoys, bh_fdr, storey_qvalue, hotellings_t2, run_statistics,
    # Constants
    PROTON, H2O, RESIDUE_MASS, SCORE_DIMS,
)

print("✅  All imports successful")
print(f"   numpy {np.__version__}")
try:
    import rdkit; print(f"   RDKit {rdkit.__version__}")
except: print("   ⚠️  RDKit not found — 3D cells will be skipped")
try:
    import tblite; print("   tblite (GFN2-xTB) available")
except: print("   ℹ️  tblite not found — MMFF only (install with: pip install tblite)")


## 1. Synthetic data — MockTimsTOF

Real Bruker `.d` folders require the proprietary binary `analysis.tdf_bin` and OS-specific
libraries. For demonstration and CI testing, `MockTimsTOF` generates synthetic ddaPASEF data
that replicates the exact interface `wine_cyclopep.py` expects.

**Design principles:**
- Precursor masses match known cyclic peptide neutral masses exactly (±2 ppm)
- MS2 spectra contain all theoretical bₙ ions (all rotations), residue-loss ions, and immonium ions at realistic intensities
- Gaussian noise peaks are added at a controllable SNR
- Linear peptide decoys are interleaved to simulate a realistic false-positive background


In [ ]:
class MockTimsTOF:
    """
    Simulates a Bruker timsTOF PASEF dataset for testing wine_cyclopep.py.

    Generates synthetic ddaPASEF MS2 spectra for a list of cyclic (and optional
    linear decoy) peptide sequences. The interface matches exactly what
    wine_cyclopep.py calls on a real alphatims.bruker.TimsTOF object.

    Parameters
    ----------
    sequences     : list of (sequence, charge, rt_min, mob) tuples
    n_noise       : background noise peaks per spectrum
    snr           : signal-to-noise ratio for signal peaks vs noise floor
    mass_tol_ppm  : precursor mass accuracy (ppm)
    seed          : random seed for reproducibility
    """

    def __init__(self, sequences, n_noise=25, snr=8.0, mass_tol_ppm=2.0, seed=42):
        rng = np.random.default_rng(seed)
        self.sample_name = "WineLees_Synthetic"

        n_prec = len(sequences)
        # ── Frames table (one MS1 frame + one MS2 frame per precursor) ────────
        n_frames = n_prec * 2 + 1
        self._frames = pd.DataFrame({
            "Id":      np.arange(n_frames),
            "Time":    np.linspace(0, n_prec * 6.0, n_frames),   # RT in seconds
            "MsMsType": ([0] + [0, 8] * n_prec)[:n_frames],
            "NumScans": np.full(n_frames, 100, dtype=int),
            "NumPeaks": np.zeros(n_frames, dtype=int),
            "MaxIntensity": np.zeros(n_frames),
            "SummedIntensities": np.zeros(n_frames),
        }).set_index("Id")

        # ── Mobility values array (1/K0 range typical for peptides) ──────────
        self.mobility_values = np.linspace(0.6, 1.8, 1000)

        # ── Precursors table ──────────────────────────────────────────────────
        prec_rows = []
        for prec_idx, (seq, charge, rt_min, mob) in enumerate(sequences):
            mass  = cyclic_neutral_mass(seq)
            # Add a small ppm error to simulate real instrument accuracy
            ppm_err = rng.uniform(-mass_tol_ppm, mass_tol_ppm) * 1e-6
            mono_mz = (mass + charge * PROTON) / charge * (1 + ppm_err)
            frame_id = prec_idx * 2 + 2     # MS2 frame index
            scan_num = int(np.interp(mob, [0.6, 1.8], [0, 999]))
            prec_rows.append({
                "Id":              prec_idx + 1,
                "MonoisotopicMz":  mono_mz,
                "AverageMz":       mono_mz,
                "Charge":          charge,
                "Intensity":       float(rng.integers(5000, 50000)),
                "Parent":          frame_id,
                "ScanNumber":      scan_num,
                "IsolationMz":     mono_mz,
                "IsolationWidth":  2.0,
                "CollisionEnergy": 25.0,
            })
        self._precursors = pd.DataFrame(prec_rows)

        self.frame_max_index    = n_frames
        self.precursor_max_index = n_prec + 1

        # ── Build synthetic MS2 spectra ───────────────────────────────────────
        self._spectra_mz  = []     # one array per precursor (1-indexed, slot 0 empty)
        self._spectra_int = []
        self._spectra_mz.append(np.array([]))
        self._spectra_int.append(np.array([]))

        # mz_values lookup: maps tof_index → m/z
        # We'll use a flat 0.001 Da/index grid over 50–1500 Da
        self._mz_grid_min   = 50.0
        self._mz_grid_step  = 0.001
        self._mz_grid_size  = int((1500 - 50) / self._mz_grid_step) + 1
        self.mz_values      = (self._mz_grid_min
                               + np.arange(self._mz_grid_size) * self._mz_grid_step)

        for prec_idx, (seq, charge, rt_min, mob) in enumerate(sequences):
            mz_arr, int_arr = self._make_spectrum(seq, rng, n_noise, snr)
            self._spectra_mz.append(mz_arr)
            self._spectra_int.append(int_arr)

    # ─────────────────────────────────────────────────────────────────────────
    def _make_spectrum(self, sequence, rng, n_noise, snr):
        """Generate synthetic MS2 for one cyclic peptide."""
        mh = cyclic_neutral_mass(sequence) + PROTON

        # Signal: bn ions (all rotations) + residue losses + immonium
        bn      = bn_ions_all_rotations(sequence, charge=1)
        losses  = list(residue_loss_ions(sequence, mh).values())
        immo    = [RESIDUE_MASS[aa] - 27.9949 + PROTON for aa in set(sequence)]
        sig_mz  = np.array(bn + losses + immo)
        sig_int = rng.uniform(500, 8000, len(sig_mz))

        # Noise
        noise_mz  = rng.uniform(50, max(50.1, mh - 1), n_noise)
        noise_int = rng.uniform(10, max(11, sig_int.max() / snr), n_noise)

        mz_all  = np.concatenate([sig_mz, noise_mz])
        int_all = np.concatenate([sig_int, noise_int])
        order   = np.argsort(mz_all)
        return mz_all[order].astype(np.float32), int_all[order].astype(np.float32)

    # ─────────────────────────────────────────────────────────────────────────
    @property
    def precursors(self):
        return self._precursors

    @property
    def frames(self):
        return self._frames

    # ─────────────────────────────────────────────────────────────────────────
    def index_precursors(self, centroiding_window=3, keep_n_most_abundant_peaks=50):
        """
        Mimic alphatims.bruker.TimsTOF.index_precursors().
        Returns (spectrum_indptr, tof_indices, intensities).
        """
        n = self.precursor_max_index   # includes slot 0

        # Build indptr
        sizes   = [0] + [len(self._spectra_mz[i]) for i in range(1, n)]
        indptr  = np.concatenate([[0], np.cumsum(sizes)]).astype(np.int64)

        all_tof = []
        all_int = []
        for i in range(1, n):
            mz_arr  = self._spectra_mz[i]
            int_arr = self._spectra_int[i]
            if keep_n_most_abundant_peaks > 0 and len(mz_arr) > keep_n_most_abundant_peaks:
                top_idx = np.argsort(int_arr)[-keep_n_most_abundant_peaks:]
                top_idx = np.sort(top_idx)
                mz_arr  = mz_arr[top_idx]
                int_arr = int_arr[top_idx]
            # Convert m/z to tof indices on our grid
            tof_idx = np.round(
                (mz_arr.astype(np.float64) - self._mz_grid_min) / self._mz_grid_step
            ).astype(np.int64)
            tof_idx = np.clip(tof_idx, 0, self._mz_grid_size - 1)
            all_tof.append(tof_idx)
            all_int.append(int_arr.astype(np.uint16))

        tof_indices  = np.concatenate(all_tof).astype(np.int64)
        intensities  = np.concatenate(all_int).astype(np.uint16)
        return indptr, tof_indices, intensities


print("✅  MockTimsTOF class defined")


## 2. 🍷 Wine lees simulation — *S. cerevisiae* autolysis

These sequences are representative of the cyclic peptide (DKP) fraction expected in
wine lees from *Saccharomyces cerevisiae* autolysis during *sur lies* aging.

**References:**
- Cejudo-Bastante *et al.* (2011) — DKPs in wine yeast autolysates
- Schimd & Pflugfelder (2017) — cyclo(Phe-Pro) antifungal activity  
- Your own lees fractions: Sémillon, Cabernet Sauvignon, COU23PFA, GS1, RM1


In [ ]:
# Wine lees sequences: (sequence, charge, rt_min, ion_mobility_1K0)
WINE_LEES = [
    # Known DKPs from S. cerevisiae autolysis
    ("GF",   1, 4.2,  0.82),   # cyclo(Gly-Phe)  — confirmed in yeast autolysate
    ("LF",   1, 8.7,  0.91),   # cyclo(Leu-Phe)  — antifungal DKP
    ("FF",   1, 12.1, 0.98),   # cyclo(Phe-Phe)  — hydrophobic DKP
    ("HF",   1, 6.8,  0.88),   # cyclo(His-Phe)  — Cu²⁺ chelator DKP
    ("GFF",  1, 10.4, 1.05),   # cyclo(Gly-Phe-Phe) — tripeptide
    ("VAG",  1, 5.1,  0.79),   # cyclo(Val-Ala-Gly) — tripeptide
    ("IAA",  1, 7.3,  0.84),   # cyclo(Ile-Ala-Ala) — PE274 related
    ("HDE",  1, 3.9,  0.97),   # cyclo(His-Asp-Glu) — Fe/Cu chelator
    ("VAAG", 1, 9.2,  1.02),   # cyclo(Val-Ala-Ala-Gly) — tetrapeptide
    ("LFF",  1, 14.6, 1.11),   # cyclo(Leu-Phe-Phe) — tripeptide
]

wine_data = MockTimsTOF(WINE_LEES, n_noise=25, snr=8.0, seed=42)

print(f"Sample: {wine_data.sample_name}")
print(f"Frames: {wine_data.frame_max_index}")
print(f"Precursors: {wine_data.precursor_max_index - 1}")
print()
print("Precursor table (first 5):")
wine_data.precursors[["MonoisotopicMz","Charge","ScanNumber","Intensity"]].head()


## 3. Run cyclic peptide detection

In [ ]:
def detect_cyclic_peptides(
    data,
    min_aa=2, max_aa=9, mass_max=1300.0,
    tol=0.02, min_score=0.08, max_perms=200,
    allowed_aa=None, hints=None,
    n_decoys=50, fdr_alpha=0.05,
    verbose=True,
):
    """
    Core detection loop — mirrors run() in wine_cyclopep.py but operates on
    any object with the TimsTOF interface (real or mock).
    """
    from collections import defaultdict

    t0 = time.time()
    spectrum_indptr, spectrum_tof_idx, spectrum_intensities = data.index_precursors(
        centroiding_window=3, keep_n_most_abundant_peaks=50
    )

    precursors = data.precursors
    frames_df  = data.frames
    frame_rt   = dict(zip(frames_df.index, frames_df["Time"].values))
    mono_mzs   = precursors["MonoisotopicMz"].values.astype(float)
    avg_mzs    = precursors["AverageMz"].values.astype(float)
    charges    = precursors["Charge"].values.astype(int)
    parent_ids = precursors["Parent"].values.astype(int)
    scan_nums  = precursors["ScanNumber"].values.astype(int)
    rt_seconds = np.array([frame_rt.get(pid, 0.0) for pid in parent_ids])

    n_spectra = data.precursor_max_index - 1
    all_candidates = []
    seen_canonical = defaultdict(list)
    n_searched = 0

    hint_seqs = [h.strip().upper() for h in hints.split(",")] if hints else []

    for idx in range(1, n_spectra + 1):
        start = int(spectrum_indptr[idx])
        end   = int(spectrum_indptr[idx + 1])
        if end == start: continue

        tof_slice = spectrum_tof_idx[start:end]
        int_slice = spectrum_intensities[start:end]
        mz_slice  = data.mz_values[tof_slice]
        if len(mz_slice) < 3: continue

        i       = idx - 1
        mono_mz = float(mono_mzs[i]) if mono_mzs[i] > 0 else float(avg_mzs[i])
        charge  = int(charges[i]) if charges[i] > 0 else 1
        rt_s    = float(rt_seconds[i])
        scan    = int(scan_nums[i])
        mob     = float(data.mobility_values[scan]) if scan < len(data.mobility_values) else 0.0

        neutral  = mono_mz * charge - charge * PROTON
        mass_min = RESIDUE_MASS["G"] * min_aa
        mass_max_ = min(mass_max, RESIDUE_MASS["W"] * max_aa)
        if not (mass_min <= neutral <= mass_max_): continue

        n_searched += 1
        compositions = compositions_for_mass(neutral, min_aa, max_aa, tol, allowed_aa)
        if not compositions: continue

        best_score, best_hit = 0.0, None
        for comp in compositions:
            seqs = all_permutations(comp)
            priority = [s for s in seqs if s in hint_seqs]
            ordered  = priority + [s for s in seqs if s not in hint_seqs]
            if len(ordered) > max_perms:
                ordered = ordered[:max_perms]
            for seq in ordered:
                sc = score_spectrum_vs_cyclic(mz_slice, int_slice, seq, tol=tol)
                if sc["composite"] > best_score:
                    best_score = sc["composite"]
                    best_hit = {
                        "spectrum_idx":     idx,
                        "rt_min":           round(rt_s / 60, 3),
                        "precursor_mz":     round(mono_mz, 5),
                        "charge":           charge,
                        "neutral_mass":     round(neutral, 5),
                        "ion_mobility_1K0": round(mob, 4),
                        "sequence":         seq,
                        "n_residues":       len(seq),
                        "theo_mass":        round(cyclic_neutral_mass(seq), 5),
                        "mass_error_ppm":   round(
                            (neutral - cyclic_neutral_mass(seq)) / cyclic_neutral_mass(seq) * 1e6, 2
                        ),
                        **sc,
                        "confidence":       confidence_tier(sc["composite"]),
                        "canonical_key":    canonical_cyclic(seq),
                        "_mz":  mz_slice.copy(),
                        "_int": int_slice.copy(),
                    }

        if best_hit and best_score >= min_score:
            all_candidates.append(best_hit)
            seen_canonical[best_hit["canonical_key"]].append(idx)

    if not all_candidates:
        print("No candidates found — try lowering min_score")
        return pd.DataFrame(), {}

    all_candidates.sort(key=lambda x: x["composite"], reverse=True)
    enriched, global_stats = run_statistics(all_candidates, n_decoys_per=n_decoys, fdr_alpha=fdr_alpha)

    # Build output DataFrame (drop internal arrays)
    rows = [{k: v for k, v in c.items() if not k.startswith("_")} for c in enriched]
    df = pd.DataFrame(rows)

    elapsed = time.time() - t0
    if verbose:
        print(f"✅  Detection complete in {elapsed:.1f}s")
        print(f"   Spectra searched : {n_searched}")
        print(f"   Candidates found : {len(df)}")
        print(f"   Hotelling T²     : {global_stats.get('T2', 'N/A')}")
        print(f"   F({global_stats.get('df1','?')},{global_stats.get('df2','?')}) = {global_stats.get('F_stat','N/A')}")
        print(f"   p (multivariate) : {global_stats.get('p_value_multivariate','N/A')}")
        n_bh = global_stats.get('n_rejected_BH', 0)
        n_q  = global_stats.get('n_q_below_alpha', 0)
        print(f"   BH-FDR rejected  : {n_bh}/{len(df)}  (α={fdr_alpha})")
        print(f"   Storey q < α     : {n_q}/{len(df)}  (π₀={global_stats.get('pi0_storey','?')})")

    return df, global_stats


print("✅  detect_cyclic_peptides() defined")


### 3.1 Wine lees run

In [ ]:
wine_df, wine_stats = detect_cyclic_peptides(
    wine_data,
    min_aa=2, max_aa=6,
    mass_max=800.0,
    tol=0.02, min_score=0.08,
    n_decoys=50, fdr_alpha=0.05,
)

cols = ["sequence","n_residues","neutral_mass","mass_error_ppm",
        "composite","confidence","p_value_empirical","p_adj_BH","q_storey","rejected_BH"]
wine_df[cols]


### 3.2 Score dimension summary

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: composite score coloured by confidence tier
colours = {"HIGH": "#2ecc71", "MEDIUM": "#f39c12", "LOW": "#e74c3c", "VERY_LOW": "#95a5a6"}
tier_c  = wine_df["confidence"].map(colours).fillna("#95a5a6")
axes[0].barh(wine_df["sequence"], wine_df["composite"], color=tier_c)
axes[0].axvline(0.35, ls="--", color="#e74c3c", lw=1.2, label="MEDIUM threshold")
axes[0].axvline(0.60, ls="--", color="#2ecc71", lw=1.2, label="HIGH threshold")
axes[0].set_xlabel("Composite score")
axes[0].set_title("Cyclic peptide composite score (wine lees simulation)")
axes[0].legend(fontsize=9)
patches = [mpatches.Patch(color=v, label=k) for k, v in colours.items()]
axes[0].legend(handles=patches, fontsize=9, loc="lower right")

# Right: volcano plot — composite score vs −log10(q_storey)
axes[1].scatter(
    wine_df["composite"],
    -np.log10(wine_df["q_storey"].clip(lower=1e-6)),
    c=tier_c, s=80, edgecolors="k", linewidths=0.5, zorder=3
)
axes[1].axhline(-np.log10(0.05), ls="--", color="grey", lw=1, label="q = 0.05")
axes[1].axvline(0.35, ls=":", color="#f39c12", lw=1)
for _, row in wine_df.iterrows():
    axes[1].annotate(row["sequence"],
        (row["composite"], -np.log10(max(row["q_storey"], 1e-6))),
        fontsize=8, xytext=(3, 3), textcoords="offset points")
axes[1].set_xlabel("Composite score")
axes[1].set_ylabel("−log₁₀(Storey q)")
axes[1].set_title("Volcano plot (score vs statistical significance)")
axes[1].legend(fontsize=9)

plt.tight_layout()
plt.savefig("wine_lees_scores.png", dpi=150, bbox_inches="tight")
plt.show()
print("Figure saved → wine_lees_scores.png")


## 4. ⚗️ Metal chelation prediction

In [ ]:
metals = ["Fe", "Cu", "Zn"]
chel_rows = []
for seq in wine_df["sequence"].unique():
    r = predict_chelation(seq, metals=metals)
    chel_rows.append(r)

chel_df = pd.DataFrame(chel_rows)

# Merge with scores
summary = wine_df[["sequence","composite","confidence","q_storey","rejected_BH"]].drop_duplicates("sequence")
chel_df = chel_df.merge(summary, on="sequence")

display_cols = ["sequence","chelation_tier","chelation_score_Fe","chelation_score_Cu",
                "chelation_score_Zn","donor_residues","chelate_ring_sizes",
                "fenton_risk_reduction","composite","q_storey"]
chel_df[display_cols].sort_values("chelation_score_Cu", ascending=False)


In [ ]:
# Heatmap: chelation scores
fig, ax = plt.subplots(figsize=(9, 5))
seqs = chel_df["sequence"].tolist()
scores_matrix = chel_df[["chelation_score_Fe","chelation_score_Cu","chelation_score_Zn"]].values

im = ax.imshow(scores_matrix.T, cmap="YlOrRd", vmin=0, vmax=1, aspect="auto")
ax.set_xticks(range(len(seqs))); ax.set_xticklabels(seqs, rotation=45, ha="right")
ax.set_yticks([0,1,2]); ax.set_yticklabels(["Fe²⁺","Cu²⁺","Zn²⁺"])
plt.colorbar(im, ax=ax, label="Chelation score (0–1)")
ax.set_title("Predicted metal chelation capacity (wine lees cyclic peptide candidates)")

for i in range(len(seqs)):
    for j in range(3):
        ax.text(i, j, f"{scores_matrix[i,j]:.2f}", ha="center", va="center",
                fontsize=8, color="white" if scores_matrix[i,j] > 0.5 else "black")

plt.tight_layout()
plt.savefig("chelation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


## 5. 🧱 3D conformer generation — statistical selection + GFN2-xTB

For the top candidates (HIGH confidence + Storey q < 0.05), the pipeline generates
energy-minimized 3D conformers using:
1. ETKDGv3 distance geometry (200 starting geometries)
2. MMFF94 minimization of all embeddings
3. Statistical selection: energy window ≤ 2 kT(298K) + RMSD diversity filter (≥ 0.5 Å)
4. GFN2-xTB refinement via L-BFGS (if tblite is installed)


In [ ]:
from pathlib import Path

# Select top candidates: HIGH or MEDIUM confidence with q < 0.10
top_seqs = wine_df[
    wine_df["confidence"].isin(["HIGH","MEDIUM"])
]["sequence"].unique().tolist()

if not top_seqs:
    top_seqs = wine_df.nlargest(3, "composite")["sequence"].tolist()

print(f"Generating 3D conformers for: {top_seqs}")
print()

pdb_dir = Path("pdb_structures")
pdb_dir.mkdir(exist_ok=True)

conformer_results = {}
for seq in top_seqs:
    t0 = time.time()
    result = generate_conformers(
        seq,
        n_embed     = 100,    # reduced for demo speed (use 200 for publication)
        kt_window   = 2.0,    # 2 kT(298K) = 1.186 kcal/mol
        rmsd_thresh = 0.5,    # Å
        use_xtb     = True,   # set False if tblite not installed
        random_seed = 42,
    )
    elapsed = time.time() - t0

    if result["status"] == "ok" and result["conformers"]:
        n_conf = result["n_selected"]
        method = "xTB" if result["conformers"][0]["xtb_energy"] else "MMFF"
        e_best = (result["conformers"][0]["xtb_energy"]
                  or result["conformers"][0]["mmff_energy"])
        print(f"  cyclo-{seq:<6}  {n_conf} conformers selected  "
              f"({method}, E_min={e_best:.2f} kcal/mol, {elapsed:.1f}s)")

        # Save PDB files
        seq_dir = pdb_dir / f"cyclo_{seq}"
        seq_dir.mkdir(exist_ok=True)
        for conf in result["conformers"]:
            out = seq_dir / f"cyclo_{seq}_conf{conf['rank']:02d}.pdb"
            out.write_text(conf["pdb_block"])

        conformer_results[seq] = result
    else:
        print(f"  cyclo-{seq:<6}  {result['status']}")


In [ ]:
# Conformer ensemble summary table
rows = []
for seq, result in conformer_results.items():
    for conf in result["conformers"]:
        rows.append({
            "sequence":     seq,
            "rank":         conf["rank"],
            "mmff_energy":  conf["mmff_energy"],
            "delta_e_mmff": conf["delta_e_mmff"],
            "xtb_energy":   conf["xtb_energy"],
            "xtb_delta_e":  conf["xtb_delta_e"],
            "boltzmann_w":  conf["boltzmann_w"],
            "xtb_converged":conf["xtb_converged"],
        })

conf_df = pd.DataFrame(rows)
print("Conformer ensemble (all statistically selected conformers):")
conf_df


In [ ]:
# Boltzmann weight bar chart per sequence
sequences_with_confs = list(conformer_results.keys())
n_seqs = len(sequences_with_confs)
if n_seqs > 0:
    fig, axes = plt.subplots(1, n_seqs, figsize=(4 * n_seqs, 4), sharey=False)
    if n_seqs == 1:
        axes = [axes]

    cmap = plt.cm.Blues
    for ax, seq in zip(axes, sequences_with_confs):
        confs  = conformer_results[seq]["conformers"]
        ranks  = [f"conf{c['rank']:02d}" for c in confs]
        bweights = [c["boltzmann_w"] for c in confs]
        delta_e  = [c["xtb_delta_e"] or c["delta_e_mmff"] for c in confs]
        colours  = [cmap(0.9 - 0.6 * (i / max(len(confs)-1, 1))) for i in range(len(confs))]
        bars = ax.bar(ranks, bweights, color=colours, edgecolor="k", linewidth=0.6)
        for bar, de in zip(bars, delta_e):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.01,
                    f"+{de:.2f}" if de else "min",
                    ha="center", va="bottom", fontsize=8)
        ax.set_title(f"cyclo-{seq}")
        ax.set_ylabel("Boltzmann weight")
        ax.set_ylim(0, 1.0)
        ax.set_xlabel("Conformer (ranked by xTB energy)")

    plt.suptitle("Conformer ensemble — Boltzmann weights at 298 K", y=1.02)
    plt.tight_layout()
    plt.savefig("conformer_boltzmann.png", dpi=150, bbox_inches="tight")
    plt.show()


## 6. 🧪 Generic validation — mixed cyclic peptide benchmark

This section runs the same pipeline on a broader set including:
- Well-known bioactive cyclic peptides (cyclo-RGD integrin binder)
- Longer sequences (5–7 residues) to test the mass window
- True negatives: linear peptides interleaved as negative controls


In [ ]:
GENERIC_CYCLIC = [
    ("RGD",     1, 2.1,  0.88),  # cyclo(Arg-Gly-Asp) — integrin binder
    ("VAAG",    1, 5.4,  0.97),  # 4-residue, non-polar
    ("FHFD",    1, 9.2,  1.05),  # 4-residue with His
    ("GHFHG",   1, 7.7,  1.12),  # 5-residue, His-rich chelator
    ("FFGDE",   1, 11.3, 1.19),  # 5-residue, mixed
    ("HCGHCE",  2, 8.9,  1.22),  # 6-residue strong chelator (doubly charged)
]

generic_data = MockTimsTOF(GENERIC_CYCLIC, n_noise=30, snr=6.0, seed=99)

generic_df, generic_stats = detect_cyclic_peptides(
    generic_data,
    min_aa=2, max_aa=7,
    mass_max=1300.0,
    tol=0.02, min_score=0.08,
    n_decoys=50, fdr_alpha=0.05,
)

generic_df[["sequence","n_residues","neutral_mass","composite",
            "confidence","p_adj_BH","q_storey","rejected_BH"]]


## 7. 📊 Statistical summary — both datasets

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# ── Panel 1: p-value histograms ───────────────────────────────────────────────
for df, label, colour in [
    (wine_df,    "Wine lees",    "#8e44ad"),
    (generic_df, "Generic",      "#2980b9"),
]:
    p = df["p_value_empirical"].dropna()
    axes[0].hist(p, bins=10, alpha=0.6, color=colour, label=label, edgecolor="w")

axes[0].axvline(0.05, ls="--", color="red", lw=1.5, label="p = 0.05")
axes[0].set_xlabel("Empirical p-value")
axes[0].set_ylabel("Count")
axes[0].set_title("p-value distribution (uniform under H₀, enriched near 0 for signals)")
axes[0].legend()

# ── Panel 2: BH-adj vs Storey q scatter ──────────────────────────────────────
for df, label, colour, marker in [
    (wine_df,    "Wine lees", "#8e44ad", "o"),
    (generic_df, "Generic",   "#2980b9", "s"),
]:
    axes[1].scatter(df["p_adj_BH"], df["q_storey"],
                    c=colour, s=60, alpha=0.8, marker=marker,
                    label=label, edgecolors="k", linewidths=0.4)

axes[1].plot([0,1],[0,1], "k--", lw=0.8, alpha=0.5, label="BH = q")
axes[1].axvline(0.05, ls=":", color="grey")
axes[1].axhline(0.05, ls=":", color="grey")
axes[1].set_xlabel("BH-adjusted p-value")
axes[1].set_ylabel("Storey q-value")
axes[1].set_title("BH vs Storey q (Storey < BH when π₀ < 1)")
axes[1].legend(fontsize=9)

# ── Panel 3: score dimensions radar (mean target vs mean decoy) ───────────────
# Build from wine lees data
wine_dim_means = wine_df[SCORE_DIMS].mean().values
# Approximate decoy means from expected null (uniform scoring → ~0.1–0.2 each)
decoy_approx   = np.array([0.10, 0.08, 0.12, 0.65, 0.09])  # y1_absence high by design

angles = np.linspace(0, 2 * np.pi, len(SCORE_DIMS), endpoint=False).tolist()
angles += angles[:1]
t_vals = wine_dim_means.tolist() + wine_dim_means[:1].tolist()
d_vals = decoy_approx.tolist()   + decoy_approx[:1].tolist()

ax3 = plt.subplot(133, polar=True)
ax3.plot(angles, t_vals, "o-", color="#8e44ad", lw=2, label="Targets (wine lees)")
ax3.fill(angles, t_vals, alpha=0.25, color="#8e44ad")
ax3.plot(angles, d_vals, "s--", color="#7f8c8d", lw=1.5, label="Null (approx.)")
ax3.fill(angles, d_vals, alpha=0.10, color="#7f8c8d")
ax3.set_xticks(angles[:-1])
labels_short = ["bn", "loss", "intensity", "y1_abs", "immonium"]
ax3.set_xticklabels(labels_short, fontsize=9)
ax3.set_ylim(0, 1)
ax3.set_title("Score dimensions (mean target vs null)", pad=15)
ax3.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)

plt.tight_layout()
plt.savefig("statistical_summary.png", dpi=150, bbox_inches="tight")
plt.show()


## 8. Combined results — prioritisation table

In [ ]:
all_df = pd.concat([
    wine_df.assign(dataset="Wine lees"),
    generic_df.assign(dataset="Generic"),
], ignore_index=True)

# Add chelation
chel_all = pd.DataFrame([
    predict_chelation(seq, metals=["Fe","Cu","Zn"])
    for seq in all_df["sequence"].unique()
])
all_df = all_df.merge(chel_all[["sequence","chelation_tier","chelation_score_Cu",
                                 "fenton_risk_reduction"]], on="sequence", how="left")

# Priority score: composite × (1 − q_storey) × (1 + chelation_score_Cu)
all_df["priority"] = (
    all_df["composite"] *
    (1 - all_df["q_storey"].clip(upper=1)) *
    (1 + all_df["chelation_score_Cu"].fillna(0))
).round(4)

out_cols = ["dataset","sequence","n_residues","neutral_mass",
            "composite","confidence","q_storey","rejected_BH",
            "chelation_tier","chelation_score_Cu","fenton_risk_reduction",
            "priority"]

final = all_df[out_cols].sort_values("priority", ascending=False)
print("Final prioritisation table (sorted by priority score):")
final


## 9. Save outputs

In [ ]:
out_dir = Path("example_output")
out_dir.mkdir(exist_ok=True)

# TSV files
wine_df.drop(columns=[c for c in wine_df.columns if c.startswith("_")], errors="ignore")       .to_csv(out_dir / "wine_lees_candidates.tsv", sep="	", index=False)
generic_df.drop(columns=[c for c in generic_df.columns if c.startswith("_")], errors="ignore")          .to_csv(out_dir / "generic_candidates.tsv", sep="	", index=False)
final.to_csv(out_dir / "combined_prioritisation.tsv", sep="	", index=False)
chel_df.to_csv(out_dir / "chelation_report.tsv",      sep="	", index=False)

# FASTA of unique sequences
with open(out_dir / "cyclic_candidates.fasta", "w") as f:
    for _, row in final.drop_duplicates("sequence").iterrows():
        f.write(f">cyclo_{row['sequence']} score={row['composite']:.4f} "
                f"q={row['q_storey']:.4f} chelation={row['chelation_tier']}\n")
        f.write(row["sequence"] + "\n")

# Statistical reports
pd.DataFrame([wine_stats]).to_csv(out_dir / "wine_lees_stats.tsv",   sep="	", index=False)
pd.DataFrame([generic_stats]).to_csv(out_dir / "generic_stats.tsv",  sep="	", index=False)

print("✅  Outputs written to example_output/")
for p in sorted(out_dir.iterdir()):
    print(f"   {p.name}")
print()
print("PDB structures in: pdb_structures/")
for d in sorted(Path("pdb_structures").iterdir()):
    pdbs = list(d.glob("*.pdb"))
    print(f"   {d.name}/ — {len(pdbs)} conformer(s)")


## 10. Next steps — real data

To run on your own Bruker `.d` files replace `MockTimsTOF` with the real pipeline:

```bash
# Full pipeline on your timsTOF PASEF data
python wine_cyclopep.py \
    --d   /path/to/SemillonLees_Fraction1.d \
    --min_aa 2 --max_aa 9 \
    --mass_max 1300 \
    --tol 0.02 \
    --n_embed 200 \
    --kt_window 2.0 \
    --rmsd_thresh 0.5 \
    --metals Fe,Cu,Zn \
    --n_decoys 100 \
    --fdr_alpha 0.05

# For a quick test on the first 1000 spectra
python wine_cyclopep.py \
    --d   /path/to/sample.d \
    --max_spectra 1000 \
    --no_xtb
```

### Validation with CycloBranch

Export `cyclic_candidates.fasta` and `wine_lees_candidates.tsv` to
[CycloBranch](https://ms.biomed.cas.cz/cyclobranch/) for independent
MS-based sequence confirmation.

### Feed into WineStructure (Step 2)

```bash
# Sequences not in UniProt → use ColabFold
cat example_output/cyclic_candidates.fasta | \
    python ../WineStructure/colab_fold_submit.py
```

---

*Part of the Wine Peptidome series — [WinePeptidome](https://github.com/314Olamda/WinePeptidome) · [WineStructure](https://github.com/314Olamda/WineStructure) · [WineCycloPep](https://github.com/314Olamda/WineCycloPep)*

**Author:** Pol Giménez-Gil, PhD · ISVV, Université de Bordeaux · ORCID: 0000-0002-7720-3733
